In [1]:
import sqlite3
import pandas as pd
import plotly.express as px

In [2]:
conn = sqlite3.connect("alzheimers.db")
oasis1 = pd.read_csv("oasis1_snapshot.csv")
oasis2 = pd.read_csv("oasis2_longitudinal.csv")

In [3]:
oasis1.to_sql("oasis1", conn, if_exists='replace',index = False)
oasis2.to_sql("oasis2", conn, if_exists='replace',index = False)


373

In [4]:
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))

     name
0  oasis1
1  oasis2


In [ ]:
#queries 
query = """
SELECT "group", 
       COUNT(*) AS total_patients, 
       ROUND(AVG(nwbv), 4) AS avg_brain_volume, 
       ROUND(AVG(mmse), 1) AS avg_mmse, 
       ROUND(AVG(age), 1) AS avg_age
FROM oasis1
GROUP BY "group"
ORDER BY avg_brain_volume;       
"""

q1 = pd.read_sql_query(query,conn)
print("\n brain volume by group: ")
print(q1)


 brain volume by group: 
         group  total_patients  avg_brain_volume  avg_mmse  avg_age
0     Demented             100            0.7220      24.3     76.8
1  Nondemented              98            0.7525      29.0     75.9


In [7]:
# chart 1 — brain volume by group
fig1 = px.bar(
    q1,
    x='group',
    y='avg_brain_volume',
    color='group',
    text='avg_brain_volume',
    title='Average Brain Volume by Dementia Status — OASIS-1',
    labels={
        'group': 'Patient Group',
        'avg_brain_volume': 'Average Brain Volume (nWBV)'},
    color_discrete_map={
        'Nondemented': '#1D9E75',
        'Demented': '#D85A30'}
)
fig1.update_traces(textposition='outside')
fig1.update_layout(showlegend=False)
fig1.show()

In [8]:
query = """
SELECT "group", 
        ROUND(AVG(nwbv),4) AS avg_brain_volume, 
        ROUND(AVG(mmse),4) AS avg_mmse
FROM oasis1
GROUP BY "group"
ORDER BY avg_mmse;
"""
q2 = pd.read_sql_query(query,conn)
print("\n mmse and brain volume by group")
print(q2)


 mmse and brain volume by group
         group  avg_brain_volume  avg_mmse
0     Demented            0.7220   24.3200
1  Nondemented            0.7525   28.9592


In [12]:
query = """
SELECT "group",
       ROUND(AVG(age),1) AS avg_age,
       MAX(age) AS maximum_age,
       MIN(age) AS minimum_age
FROM oasis1
GROUP BY "group";       
"""
q3 = pd.read_sql_query(query,conn)
print("\n age by group")
print(q3)


 age by group
         group  avg_age  maximum_age  minimum_age
0     Demented     76.8           96           62
1  Nondemented     75.9           94           60


In [19]:
query = """
SELECT COUNT(*) AS total_patients,
       ROUND(AVG(nwbv), 4) AS avg_brain_volume, 
CASE
    WHEN education <= 2 THEN 'Low'
    WHEN education <= 4 THEN 'Medium'
    WHEN education > 4 THEN 'High'
END AS education_level
FROM oasis1
WHERE "group" = "Demented"
GROUP BY education_level
ORDER BY avg_brain_volume DESC;               
"""
q4 = pd.read_sql_query(query,conn)
print("\n education vs brain volume in demented patients")
print(q4)


 education vs brain volume in demented patients
   total_patients  avg_brain_volume education_level
0              36            0.7250          Medium
1              14            0.7210            High
2              50            0.7201             Low


In [20]:
# chart 3 — education vs brain volume
fig3 = px.bar(
    q4,
    x='avg_brain_volume',
    y='education_level',
    orientation='h',
    text='avg_brain_volume',
    title='Brain Volume by Education Level — Demented Patients Only (OASIS-1)',
    labels={
        'avg_brain_volume': 'Average Brain Volume (nWBV)',
        'education_level': 'Education Level'
    },
    color='education_level',
    color_discrete_map={
        'High': '#1D9E75',
        'Medium': '#EF9F27',
        'Low': '#D85A30'
    }
)

fig3.update_traces(textposition='outside')
fig3.update_layout(showlegend=False)
fig3.show()

In [21]:
query = """
SELECT gender,
       "group",
       COUNT(*) AS total_patients,
       ROUND(AVG(nwbv), 4) AS avg_brain_volume
FROM oasis1
GROUP BY gender,"group"
ORDER BY "group", gender
""" 
q5 = pd.read_sql_query(query,conn)
print("\n gender by group:")
print(q5)


 gender by group:
  gender        group  total_patients  avg_brain_volume
0      F     Demented              59            0.7226
1      M     Demented              41            0.7211
2      F  Nondemented              72            0.7564
3      M  Nondemented              26            0.7417


In [22]:
#quering oasis 2
query = """
SELECT "group",
        visit,
        COUNT(DISTINCT patient_id) AS patient_count,
        ROUND(AVG(nwbv), 4) AS avg_brain_volume,
        ROUND(AVG(mmse), 1) AS avg_mmse
FROM oasis2
GROUP BY "group", visit
ORDER BY "group", visit        
"""

q6 = pd.read_sql_query(query,conn)
print("\n brian volume over visits by group: ")
print(q6)


 brian volume over visits by group: 
          group  visit  patient_count  avg_brain_volume  avg_mmse
0     Converted      1             14            0.7379      29.4
1     Converted      2             12            0.7282      28.0
2     Converted      3              8            0.7062      28.5
3     Converted      4              2            0.6950      28.0
4     Converted      5              1            0.6694      30.0
5      Demented      1             64            0.7244      25.3
6      Demented      2             62            0.7135      24.2
7      Demented      3             16            0.7007      24.3
8      Demented      4              3            0.6991      20.0
9      Demented      5              1            0.6761       4.0
10  Nondemented      1             72            0.7461      29.2
11  Nondemented      2             70            0.7377      29.1
12  Nondemented      3             34            0.7356      29.5
13  Nondemented      4             10 

In [24]:
# chart 4 — brain volume over time by group
fig4 = px.line(
    q6,
    x='visit',
    y='avg_brain_volume',
    color='group',
    markers=True,
    title='Brain Volume Decline Over Visits by Group — OASIS-2',
    labels={
        'visit': 'Visit Number',
        'avg_brain_volume': 'Average Brain Volume (nWBV)',
        'group': 'Patient Group'
    },
    color_discrete_map={
        'Nondemented': '#1D9E75',
        'Demented': '#D85A30',
        'Converted': '#185FA5'
    }
)

fig4.update_layout(
    xaxis=dict(tickmode='linear', tick0=1, dtick=1)
)

fig4.show()

In [ ]:
query = """
SELECT "group",
        visit,
        ROUND(AVG(nwbv), 4) AS avg_brain_volume,
        ROUND(AVG(mmse), 1) AS avg_mmse
FROM oasis2 
WHERE "group" = 'Converted' OR "group" = 'Nondemented'
GROUP BY "group", visit
ORDER BY "group", visit ;      
"""
q7 = pd.read_sql_query(query, conn)
print("\n Converted vs Nondemented progression: ")
print(q7)


 Converted vs Nondemented progression: 
         group  visit  avg_brain_volume  avg_mmse
0    Converted      1            0.7379      29.4
1    Converted      2            0.7282      28.0
2    Converted      3            0.7062      28.5
3    Converted      4            0.6950      28.0
4    Converted      5            0.6694      30.0
5  Nondemented      1            0.7461      29.2
6  Nondemented      2            0.7377      29.1
7  Nondemented      3            0.7356      29.5
8  Nondemented      4            0.7389      29.6
9  Nondemented      5            0.7533      28.8


In [27]:
# chart 5 — mmse over visits converted vs nondemented
fig5 = px.line(
    q7,
    x='visit',
    y='avg_mmse',
    color='group',
    markers=True,
    title='Cognitive Score (MMSE) Over Visits — Converted vs Nondemented',
    labels={
        'visit': 'Visit Number',
        'avg_mmse': 'Average MMSE Score',
        'group': 'Patient Group'
    },
    color_discrete_map={
        'Nondemented': '#1D9E75',
        'Converted': '#185FA5'
    }
)

fig5.update_layout(
    xaxis=dict(tickmode='linear', tick0=1, dtick=1)
)

fig5.show()

In [29]:
query = """
SELECT "group",
       COUNT(DISTINCT patient_id) AS patient_count,
       ROUND(AVG(starting_volume), 4) AS avg_starting_volume,
       ROUND(AVG(final_volume), 4) AS avg_final_volume,
       ROUND(AVG(total_decline), 4) AS avg_total_decline
FROM ( SELECT patient_id, 
       "group",
       MAX(nwbv) AS starting_volume,
       MIN(nwbv) AS final_volume,
       MAX(nwbv) - MIN(nwbv) AS total_decline
FROM oasis2
GROUP BY patient_id, "group") AS patient_summary 
GROUP BY "group"
ORDER BY AVG_total_decline DESC;
"""
q8 = pd.read_sql_query(query, conn)
print("\n total brain volume decline by group: ")
print(q8)


 total brain volume decline by group: 
         group  patient_count  avg_starting_volume  avg_final_volume  \
0    Converted             14               0.7385            0.7147   
1     Demented             64               0.7253            0.7100   
2  Nondemented             72               0.7474            0.7330   

   avg_total_decline  
0             0.0237  
1             0.0153  
2             0.0143  


In [30]:
# chart 6 — total decline per group
fig6 = px.bar(
    q8,
    x='group',
    y='avg_total_decline',
    color='group',
    text='avg_total_decline',
    title='Average Total Brain Volume Lost by Group — OASIS-2',
    labels={
        'group': 'Patient Group',
        'avg_total_decline': 'Average Total nWBV Decline'
    },
    color_discrete_map={
        'Nondemented': '#1D9E75',
        'Demented': '#D85A30',
        'Converted': '#185FA5'
    }
)

fig6.update_traces(textposition='outside')
fig6.update_layout(showlegend=False)
fig6.show()

In [ ]:
query = """
SELECT visit,
       COUNT(patient_id) AS patient_count,
       ROUND(AVG(age), 1) AS avg_age
FROM oasis2
WHERE "group" = 'Converted' AND cdr = 0.5
GROUP BY visit
ORDER BY visit;       
"""
q9 = pd.read_sql_query(query, conn)
print("\n conversion visit: ")
print(q9)



 convertion visit: 
   visit  patient_count  avg_age
0      1              1     65.0
1      2              8     78.9
2      3              7     82.3
3      4              2     88.0
4      5              1     86.0


In [33]:
query = """
SELECT patient_id,
       visit,
       mr_delay,
       age,
       cdr,
       mmse,
       ROUND(nwbv, 4) AS brain_volume
FROM oasis2
WHERE "group" = 'Converted'
ORDER BY patient_id, visit ;      
"""

q10= pd.read_sql_query(query, conn)
print("\n Converted patients visit by visit: ")
print(q10)


 Converted patients visit by visit: 
   patient_id  visit  mr_delay  age  cdr  mmse  brain_volume
0   OAS2_0018      1         0   87  0.0  30.0        0.7151
1   OAS2_0018      3       489   88  0.0  29.0        0.7134
2   OAS2_0018      4      1933   92  0.5  27.0        0.6958
3   OAS2_0020      1         0   80  0.0  29.0        0.6932
4   OAS2_0020      2       756   82  0.5  28.0        0.6766
5   OAS2_0020      3      1563   84  0.5  26.0        0.6664
6   OAS2_0031      1         0   86  0.0  30.0        0.7181
7   OAS2_0031      2       446   88  0.0  30.0        0.7195
8   OAS2_0031      3      1588   91  0.5  28.0        0.6955
9   OAS2_0041      1         0   71  0.0  27.0        0.7713
10  OAS2_0041      2       756   73  0.0  28.0        0.7677
11  OAS2_0041      3      1331   75  0.5  28.0        0.7603
12  OAS2_0054      1         0   85  0.0  29.0        0.7010
13  OAS2_0054      2       846   87  0.5  24.0        0.6827
14  OAS2_0092      1         0   83  0.0  28.0 

In [34]:
# chart 7 — individual converted patient brain volume trajectories
fig7 = px.line(
    q10,
    x='visit',
    y='brain_volume',
    color='patient_id',
    markers=True,
    title='Individual Brain Volume Trajectories — Converted Patients Only',
    labels={
        'visit': 'Visit Number',
        'brain_volume': 'Brain Volume (nWBV)',
        'patient_id': 'Patient'
    }
)

fig7.update_layout(
    xaxis=dict(tickmode='linear', tick0=1, dtick=1),
    showlegend=True
)

fig7.show()

In [35]:
conn.close()
print("connection closed")

connection closed
